# FIFA Men's World Cup 2026 Predictor
## Stage 1: Data Alignment & Sanity Check

**Author:** Kamweti Ryan Murunga  
**Objective:** Prepare the core target variables from historical World Cup data and verify the structural integrity of the 2026 Test dataset against the Fjelstul World Cup Database.

### Scope of this Notebook:
1. Load core training, testing, and submission framework files.
2. Standardize categorical target text formats (`stage_reached`) to match Zindi constraints.
3. Perform structural name-matching across historical and future data frames to protect against lookup data leakage or missing rows.

In [2]:
import os
import pandas as pd
import numpy as np

# Set explicit seeds for absolute pipeline reproducibility
SEED = 42
np.random.seed(SEED)

print("Ready")

Ready


##  Data Ingestion
In this section, we load the raw files from the challenge repository to assess shapes, data types, and initial row distributions.

In [3]:
# Load target challenge sheets from local data repository
train = pd.read_csv('data/Train.csv')
test = pd.read_csv('data/Test.csv')
sample_sub = pd.read_csv('data/SampleSubmission.csv')

print(f"[-] Train Framework Shape : {train.shape}")
print(f"[-] Test Framework Shape  : {test.shape}")
print(f"[-] Sample Sub Shape      : {sample_sub.shape}")

[-] Train Framework Shape : (489, 12)
[-] Test Framework Shape  : (48, 2)
[-] Sample Sub Shape      : (48, 3)


## Target Standardisation & Mapping
The historical `stage_reached` strings inside the training data contain legacy tournament names (e.g., 'second group stage', 'third-place match'). We construct a strict dictionary mapping to align these outcomes into the seven exact tournament dimensions required by the Zindi submission format.

In [4]:
# Explicit mapping strategy for categorical targets
stage_cleaner = {
    'group stage': 'group',
    'second group stage': 'group',
    'round of 16': 'roundof16',
    'quarter-finals': 'qf',
    'third-place match': 'sf',
    'semi-finals': 'sf',
    'final': 'runnerup'
}

# Apply mapping and handle structural missing values safely
train['cleaned_stage'] = train['stage_reached'].map(stage_cleaner).fillna('group')

print("[+] Target variable alignment complete. Verification sample:")
display(train[['stage_reached', 'cleaned_stage']].drop_duplicates().head(5))

[+] Target variable alignment complete. Verification sample:


,stage_reached,cleaned_stage
0,final,runnerup
1,group stage,group
10,semi-finals,sf
13,round of 16,roundof16
14,third-place match,sf


## Entity Verification (Train vs. Test Alignment)
To ensure our upcoming Stage 2 Feature Store can map historical performance properties onto the 2026 teams without throwing null values, we must scan for any completely new countries debutants in the 2026 Test set.

In [5]:
train_countries = set(train['country'].unique())
test_countries = set(test['country'].unique())

new_countries = test_countries - train_countries

print(f"[-] Total Unique Historical Countries: {len(train_countries)}")
print(f"[-] Total 2026 Qualified Countries   : {len(test_countries)}")
print(f"[!] 2026 Countries with zero World Cup history: {new_countries}")

[-] Total Unique Historical Countries: 85
[-] Total 2026 Qualified Countries   : 48
[!] 2026 Countries with zero World Cup history: {'Czechia', 'Cabo Verde', "Cote d'Ivoire", 'Uzbekistan', 'Jordan', 'Curacao', 'DR Congo', 'Turkiye'}


## Name alignment

In [6]:
# 1. Define a mapping dictionary to fix historical name discrepancies
historical_name_fixer = {
    'Turkey': 'Turkiye',
    'Czech Republic': 'Czechia',
    'Ivory Coast': "Cote d'Ivoire",
    'Zaire': 'DR Congo',
    # We will also clean any potential padding whitespace
}

# 2. Update the country names in your Train dataframe
train['country'] = train['country'].replace(historical_name_fixer).str.strip()

# 3. Re-run the verification check to see who is truly a debutant
train_countries_updated = set(train['country'].unique())
still_new_countries = test_countries - train_countries_updated

print(f"[-] Remaining countries with zero history: {still_new_countries}")

[-] Remaining countries with zero history: {'Curacao', 'Jordan', 'Cabo Verde', 'Uzbekistan'}
